In [1]:
import matplotlib.pyplot as plt
import torch
from sklearn.datasets import make_moons
from tqdm.auto import tqdm
from torch import Tensor, nn
import torchode as to
from torchtune.modules import RotaryPositionalEmbeddings

In [92]:
class Flow(nn.Module):
    def __init__(self, dim: int, h: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, h),
            nn.SELU(),
            nn.Linear(h, h),
            nn.SELU(),
            nn.Linear(h, h),
            nn.SELU(),
            nn.Linear(h, 1),
        )

    def forward(self, t: Tensor, x: Tensor, history: Tensor) -> Tensor:
        t = t.unsqueeze(-1)
        u_t = self.net(torch.cat([x, history, t], dim=-1))[..., 0]  # (B)
        return u_t

In [93]:
device = "cpu"
# Training

B = 256
T = 200
H = 20

flow = Flow(dim=1 + H, h=64).to(device)
optimizer = torch.optim.AdamW(flow.parameters(), 1e-6)
loss_fn = nn.MSELoss()


sigma = 1
tau = torch.linspace(0, 1, T, device=device)
num_batches = 100
num_epochs = 10

for i in tqdm(range(num_epochs)):
    sum_loss = 0
    for j in range(num_batches):
        freq = torch.rand(B, device=device) * torch.pi + torch.pi
        x = torch.sin(tau[None, :] * freq[:, None])
        k = torch.randint(H + 1, T - 1, (B,), device=device)
        t = torch.rand(B, device=device)
        x_k = torch.gather(x, dim=1, index=k[:, None]).squeeze(1)
        x_kp1 = torch.gather(x, dim=1, index=k[:, None] + 1).squeeze(1)
        mu_t = (1 - t) * x_k + t * x_kp1
        sigma_t = torch.sqrt((sigma**2) * t * (1 - t))
        x_t = torch.randn(B, device=device) * sigma_t + mu_t
        dx_t = x_kp1 - x_k
        hist_idx = k[:, None] - torch.arange(H, 0, -1, device=device).expand(B, H)
        history = torch.gather(x, dim=1, index=hist_idx)

        optimizer.zero_grad()
        loss = loss_fn(flow(t=t + k, x=x_t[:, None], history=history), dx_t)
        loss.backward()
        optimizer.step()
        sum_loss += loss.cpu().item()
    print(sum_loss / num_batches)

  0%|          | 0/10 [00:00<?, ?it/s]

1.4122066843509673
1.1345103240013124
0.8998716878890991
0.7058705163002014
0.5467812666296958
0.41396173983812334
0.31020807087421415
0.23029984831809996
0.16919332414865493
0.1243258373439312


In [94]:
with torch.no_grad():
    # Sampling
    term = to.ODETerm(flow, with_args=True)
    step_method = to.Dopri5(term=term)
    step_size_controller = to.IntegralController(atol=1e-6, rtol=1e-3, term=term)
    solver = to.AutoDiffAdjoint(step_method, step_size_controller)
    jit_solver = torch.compile(solver)

    traj = [torch.tensor(0, dtype=torch.float32, device=device)] * (H + 1)
    for k in range(200):
        x0 = traj[-1]
        history = torch.stack(traj[-(H + 1) : -1], dim=-1)
        ivp = to.InitialValueProblem(
            y0=x0[None, None],
            t_start=torch.tensor(k, dtype=torch.float32, device=device)[None],
            t_end=torch.tensor(k + 1, dtype=torch.float32, device=device)[None],
        )
        solution = jit_solver.solve(ivp, args=history[None])
        x1 = jit_solver.solve(ivp, args=history[None]).ys[-1].squeeze()
        traj.append(x1)
    # x = torch.randn(200, device=device)
    # time_steps = torch.linspace(0, 1, 5, device=device)
    # fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharex=True, sharey=True)

    # solution = jit_solver.solve(
    #     to.InitialValueProblem(
    #         y0=x.unsqueeze(0),
    #         t_eval=time_steps[None],
    #     )
    # )
    # for i, (ax, x) in enumerate(zip(axes, solution.ys[0])):
    #     ax.scatter(tau.cpu(), x.squeeze().cpu(), s=10)
    #     ax.set_title(f"t = {time_steps[i]:.2f}")

    # plt.tight_layout()
    # plt.show()


IndexError: tuple index out of range

In [80]:
solution.ys

NameError: name 'solution' is not defined